## MCORR AND CNMF
**Analysis of riboL1-GCaMP8m responses imaged with 16x objective at zoom 1x over 765x765 pixels**

#### Define parameters


In [ ]:
# Motion correction parameters
params_mcorr = \
{
  'main':
    {
        'strides': [36, 36],
        'overlaps': [24, 24],
        'max_shifts': [12, 12],
        'max_deviation_rigid': 6,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

# CNMF parameters
params_cnmf =\
{
    'main': # indicates that these are the "main" params for the CNMF algo
        {
            'fr': 20, # framerate, very important!
            'p': 1,
            'nb': 2,
            'merge_thr': 0.85,
            'rf': 20,
            'stride': 10, # "stride" for cnmf, "strides" for mcorr
            'K': 12,
            'gSig': [4, 4],
            'ssub': 1,
            'tsub': 1,
            'method_init': 'greedy_roi',
            'min_SNR': 2.0,
            'SNR_lowest':  1.5,
            'rval_thr': 0.8,
            'rval_lowest': 0.2,
            'use_cnn': True,
            'min_cnn_thr': 0.99,
            'cnn_lowest': 0.4,
            'decay_time': 0.15,
        },
    'refit': True, # If `True`, run a second iteration of CNMF
}

# Extra parameters
params_extra = \
    {
        'cleanup': False # If `True`, run cleanup after CNMF
    }

# Concatenate the two dictionaries
params = {'params_mcorr': params_mcorr, 'params_cnmf': params_cnmf, 'params_extra': params_extra}
print(params)

#### Run MCORR and CNMF

In [ ]:
import pipeline_mcorr_cnmf as preproc
from pathlib import Path
data_path = [Path('D:/Data/2P/C57_O1M2/10022023/TSeries-10022023-1529-004')]
export_path = Path('H:/Analysis/2P/C57_O1M2/10022023/run4/mesmerize/')
_ , batch_path = preproc.run_mcorr_cnmf(data_path, params, export_path)

### Load outputs

In [ ]:
# If cleanup was set to false in the params, you can load the batch file into Mesmerize:
if params['params_extra']['cleanup'] == False:
    import mesmerize_core as mc
    df = mc.load_batch(batch_path)

### Visualize with fastplotlib


In [ ]:
import fastplotlib as fpl
import numpy as np
# first item is just the raw movie
movies = [df.iloc[0].caiman.get_input_movie()]

# subplot titles
subplot_names = ["raw"]

# add all the mcorr outputs to the list
for i, row in df.iterrows():
    # Select which row to display
    # if i == 2 or i == 3:
    # add to the list of movies to plot
    movies.append(row.mcorr.get_output())

    # subplot title to show dataframe index
    subplot_names.append(f"ix {i}")

# create the widget
mcorr_iw_multiple = fpl.ImageWidget(
    data=movies,  # list of movies
    window_funcs={"t": (np.mean, 1)}, # window functions as a kwarg, this is what the slider was used for in the ready-made viz
    grid_plot_kwargs={"size": (900, 900)},
    names=subplot_names,  # subplot names used for titles
    cmap="gray" #"gnuplot2"
)

mcorr_iw_multiple.show()

In [ ]:
mcorr_iw_multiple.close()

### Visualize with `mesmerize-viz`

In [ ]:
# %gui qt

In [ ]:
viz_simple = df.cnmf.viz(
    # image_data_options=["input", "rcm"],
    image_data_options=["max"],
)
viz_simple.show(sidecar=True)
# viz_simple.show()
viz_simple.image_widget.cmap = "gray"

In [ ]:
viz_simple.close()